This was originally ran in Google co-lab, however, only relies on the very lists you can find under the semantic neighborhood folder. 
This is how the values in true_countsSub.csv and true_countsSup.csv were determined. The word pair list can be found in word data.csv.

In [ ]:
import os
import pandas as pd
import time
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key="your-api-key")  # Replace with your actual API key

# Define paths
base_path = "/content/drive/My Drive/GPT_4o_Neighborhood"
word_pairs_file = os.path.join(base_path, "word pairs.csv")
superordinate_path = os.path.join(base_path, "Superordinate")

# Read the word pairs CSV
word_pairs_df = pd.read_csv(word_pairs_file, sep=',')  # Adjust delimiter if needed

# Function to check category relationship using OpenAI API
def check_category_relationship(word, subordinate_word):
    prompt = f"Is '{word}' in the same category as '{subordinate_word}'? Respond with 'True' if they belong to the same general category, otherwise respond with 'False'."

    for attempt in range(5):  # Retry up to 5 times in case of failure
        try:
            response = client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}]
            )
            result = response.choices[0].message.content.strip()
            return result if result in ["True", "False"] else "ERROR"
        except Exception as e:
            if "rate limit" in str(e).lower():
                print("Rate limit hit. Pausing for 60 seconds...")
                time.sleep(60)  # Wait for 60 seconds before retrying
            else:
                print(f"Error processing word '{word}': {e}")
                time.sleep(5)  # Short pause before retrying other errors
    return "ERROR"  # Return ERROR after 5 failed attempts

# Iterate through each superordinate-subordinate pair
for _, row in word_pairs_df.iterrows():
    superordinate = row["Superordinate"]
    subordinate = row["Subordinate"]

    # Construct the expected file path
    superordinate_file = os.path.join(superordinate_path, f"{superordinate.lower()}.csv")

    # Skip if the superordinate file does not exist
    if not os.path.exists(superordinate_file):
        print(f"File not found: {superordinate_file}, skipping...")
        continue

    # Read the superordinate CSV (assumed to have three columns: target_word, neighbor, similarity)
    df = pd.read_csv(superordinate_file, sep=',')  # Adjust delimiter if needed

    # Ensure the necessary columns exist
    if df.shape[1] < 2:
        print(f"Skipping {superordinate_file}, not enough columns.")
        continue

    # Extract the second column (neighbor words)
    neighbors = df.iloc[:, 1].astype(str)

    # Apply the OpenAI API check
    df["Category_Sharing"] = neighbors.apply(lambda word: check_category_relationship(word, subordinate))

    # Save the modified CSV back to the same location
    df.to_csv(superordinate_file, index=False)
    print(f"Processed and updated: {superordinate_file}")

print("All files processed!")


In [ ]:
import os
import pandas as pd
import time
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key="your-api-key")  # Replace with your actual API key

# Define paths
base_path = "/content/drive/My Drive/GPT_4o_Neighborhood"
word_pairs_file = os.path.join(base_path, "word pairs.csv")
subordinate_path = os.path.join(base_path, "Subordinate")  # Changed to Subordinate folder

# Read the word pairs CSV
word_pairs_df = pd.read_csv(word_pairs_file, sep=',')  # Adjust delimiter if needed

# Function to check category relationship using OpenAI API
def check_category_relationship(subordinate_word, superordinate_word):
    prompt = f"Is '{subordinate_word}' in the category of '{superordinate_word}'? Respond with 'True' if it belongs to that category, otherwise respond with 'False'."

    for attempt in range(5):  # Retry up to 5 times in case of failure
        try:
            response = client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}]
            )
            result = response.choices[0].message.content.strip()
            return result if result in ["True", "False"] else "ERROR"
        except Exception as e:
            if "rate limit" in str(e).lower():
                print("Rate limit hit. Pausing for 60 seconds...")
                time.sleep(60)  # Wait for 60 seconds before retrying
            else:
                print(f"Error processing word '{subordinate_word}': {e}")
                time.sleep(5)  # Short pause before retrying other errors
    return "ERROR"  # Return ERROR after 5 failed attempts

# Iterate through each superordinate-subordinate pair
for _, row in word_pairs_df.iterrows():
    superordinate = row["Superordinate"]
    subordinate = row["Subordinate"]

    # Construct the expected file path
    subordinate_file = os.path.join(subordinate_path, f"{subordinate.lower()}.csv")  # Changed to subordinate file

    # Skip if the subordinate file does not exist
    if not os.path.exists(subordinate_file):
        print(f"File not found: {subordinate_file}, skipping...")
        continue

    # Read the subordinate CSV
    df = pd.read_csv(subordinate_file, sep=',')  # Adjust delimiter if needed

    # Ensure the necessary columns exist
    if df.shape[1] < 2:
        print(f"Skipping {subordinate_file}, not enough columns.")
        continue

    # Check if "Category_Sharing" column already exists and is fully populated
    if "Category_Sharing" in df.columns and not df["Category_Sharing"].isna().any() and "ERROR" not in df["Category_Sharing"].values:
        print(f"Skipping already processed file: {subordinate_file}")
        continue  # Skip this file

    # Extract the second column (neighbor words)
    neighbors = df.iloc[:, 1].astype(str)

    # Apply the OpenAI API check
    df["Category_Sharing"] = neighbors.apply(lambda word: check_category_relationship(word, superordinate))

    # Save the modified CSV back to the same location
    df.to_csv(subordinate_file, index=False)
    print(f"Processed and updated: {subordinate_file}")

print("All files processed!")
